In [24]:
import matplotlib.pyplot as plt

import torch
import torchvision

from torch import nn
from torchvision import transforms


print(torch.__version__)
print(torchvision.__version__)

2.6.0+cu126
0.21.0+cu126


In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [26]:
from pathlib import Path

data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

(WindowsPath('data/pizza_steak_sushi/train'),
 WindowsPath('data/pizza_steak_sushi/test'))

In [27]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
weights

EfficientNet_B0_Weights.IMAGENET1K_V1

In [28]:
auto_transforms = weights.transforms()
auto_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [29]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [30]:
train_data = datasets.ImageFolder(train_dir, transform=auto_transforms)
test_data = datasets.ImageFolder(test_dir, transform=auto_transforms)

In [31]:
class_names = train_data.classes

In [32]:
BATCH_SIZE = 32

In [61]:
from sympy import true


train_dataloader = DataLoader(dataset=train_data,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              pin_memory=True)

test_dataloader = DataLoader(dataset=test_data,
                              batch_size=BATCH_SIZE,
                              shuffle=False,
                              pin_memory=True)

In [62]:
from going_modular.going_modular import data_setup

train_dataloader1, test_dataloader1, class_names1 = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=auto_transforms,
                                                                               batch_size=32)

train_dataloader1, test_dataloader1, class_names1

(<torch.utils.data.dataloader.DataLoader at 0x1c822c684f0>,
 ['pizza', 'steak', 'sushi'])

In [63]:
train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x1c7a47d50c0>,
 ['pizza', 'steak', 'sushi'])

In [58]:
import torch

def compare_dataloaders(dl1, dl2):
    print("Dataloader 1:")
    print(f"  Number of batches: {len(dl1)}")
    print("Dataloader 2:")
    print(f"  Number of batches: {len(dl2)}\n")
    
    # Compare the shapes of the first batch from each
    batch1 = next(iter(dl1))
    batch2 = next(iter(dl2))
    
    inputs1, labels1 = batch1
    inputs2, labels2 = batch2
    
    print("First batch input shapes:")
    print(f"  Dataloader 1: {inputs1.shape}")
    print(f"  Dataloader 2: {inputs2.shape}\n")
    
    print("First batch label shapes:")
    print(f"  Dataloader 1: {labels1.shape}")
    print(f"  Dataloader 2: {labels2.shape}\n")
    
    # Optionally, compare label distributions in the first batch
    print("Unique labels in first batch:")
    print(f"  Dataloader 1: {torch.unique(labels1)}")
    print(f"  Dataloader 2: {torch.unique(labels2)}")

# Example usage:
compare_dataloaders(train_dataloader, train_dataloader1)

Dataloader 1:
  Number of batches: 8
Dataloader 2:
  Number of batches: 8

First batch input shapes:
  Dataloader 1: torch.Size([32, 3, 224, 224])
  Dataloader 2: torch.Size([32, 3, 224, 224])

First batch label shapes:
  Dataloader 1: torch.Size([32])
  Dataloader 2: torch.Size([32])

Unique labels in first batch:
  Dataloader 1: tensor([0, 1, 2])
  Dataloader 2: tensor([0, 1, 2])


In [59]:
import torch

def compare_dataloaders(dl1, dl2):
    print("Dataloader 1:")
    print(f"  Number of batches: {len(dl1)}")
    print("Dataloader 2:")
    print(f"  Number of batches: {len(dl2)}\n")
    
    # Compare the shapes of the first batch from each
    batch1 = next(iter(dl1))
    batch2 = next(iter(dl2))
    
    inputs1, labels1 = batch1
    inputs2, labels2 = batch2
    
    print("First batch input shapes:")
    print(f"  Dataloader 1: {inputs1.shape}")
    print(f"  Dataloader 2: {inputs2.shape}\n")
    
    print("First batch label shapes:")
    print(f"  Dataloader 1: {labels1.shape}")
    print(f"  Dataloader 2: {labels2.shape}\n")
    
    # Optionally, compare label distributions in the first batch
    print("Unique labels in first batch:")
    print(f"  Dataloader 1: {torch.unique(labels1)}")
    print(f"  Dataloader 2: {torch.unique(labels2)}")

# Example usage:
compare_dataloaders(test_dataloader, test_dataloader1)

Dataloader 1:
  Number of batches: 3
Dataloader 2:
  Number of batches: 3

First batch input shapes:
  Dataloader 1: torch.Size([32, 3, 224, 224])
  Dataloader 2: torch.Size([32, 3, 224, 224])

First batch label shapes:
  Dataloader 1: torch.Size([32])
  Dataloader 2: torch.Size([32])

Unique labels in first batch:
  Dataloader 1: tensor([0, 1])
  Dataloader 2: tensor([0, 1])


In [64]:
train_dataloader == train_dataloader1

False

In [47]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model = torchvision.models.efficientnet_b0(weights=weights).to(device)
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [48]:
for params in model.features.parameters():
    # print(params)
    params.requires_grad = False

In [49]:
from torch import nn
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=len(class_names))
).to(device)

model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=3, bias=True)
)

In [67]:
next(model.parameters()).device

device(type='cuda', index=0)

In [51]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(),
                             lr=0.001)

In [70]:
torch.manual_seed(42)

model.train()

train_loss, train_acc = 0, 0

for _ in range(5):

    for batch, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)

        print(X.device)
        print(y.device)
        print(next(model.parameters()).device)

        y_pred = model(X)

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item() 

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metric across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch 
    train_loss = train_loss / len(train_dataloader)
    train_acc = train_acc / len(train_dataloader)
    print(train_loss, train_acc)

cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
0.18959816917777061 0.98046875
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
0.20788125647231936 1.11083984375
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
0.2057555084466003 1.12713623046875
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
0.2265037436663988 1.1174545288085938
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
0.2517192958466694 1.

In [71]:
from going_modular.going_modular import engine

torch.manual_seed(42)
torch.cuda.manual_seed(42)

from timeit import default_timer as timer

start_time = timer()

results = engine.train(model=model,
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       epochs=5,
                       device=device)

end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.1645 | train_acc: 0.9883 | test_loss: 0.3292 | test_acc: 0.8864
Epoch: 2 | train_loss: 0.2191 | train_acc: 0.9766 | test_loss: 0.3175 | test_acc: 0.8759
Epoch: 3 | train_loss: 0.1829 | train_acc: 0.9844 | test_loss: 0.3397 | test_acc: 0.8759
Epoch: 4 | train_loss: 0.4179 | train_acc: 0.8516 | test_loss: 0.3362 | test_acc: 0.8352
Epoch: 5 | train_loss: 0.2929 | train_acc: 0.8516 | test_loss: 0.3732 | test_acc: 0.8561
[INFO] Total training time: 10.304 seconds


In [42]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat